# Under the Hood: Fluss-Rust & Issue #424
Welcome! This notebook provides a rigorous framework to get you up to speed with Rust, the `fluss-rust` architecture, and the specific context needed to solve **[Issue #424](https://github.com/apache/fluss-rust/issues/424)**.

## 1. Rust & PyO3 Crash Course
`fluss-rust` uses **PyO3** to create Python bindings. PyO3 allows you to write native Rust code that compiles into a Python module natively.

- `#[pyclass]`: Exposes a Rust struct as a Python class.
- `#[pymethods]`: Exposes Rust methods to Python. This is where methods like `__init__`, `__str__`, and `poll()` live.
- `PyResult<T>`: The return type for methods that might raise a Python exception.
- `future_into_py`: Converts a Rust `async` Future into a Python awaitable (`asyncio.Future`/`Task`).

## 2. Architecture of `LogScanner`
In `fluss-rust`, the python binding for `LogScanner` is defined in `bindings/python/src/table.rs` (starts around line 1903).
The current implementation forces users to manually call `poll()` to get a `ScanRecords` object (a batch of records), and then iterate over them manually.

In [ ]:
// Pseudo-code of the current LogScanner implementation
#[pyclass]
pub struct LogScanner {
    inner: ScannerKind, // Wrapper to the core Rust engine scanner
    // ... other metadata (like projections, admins)
}

#[pymethods]
impl LogScanner {
    // The manual polling function you are trying to replace with an async for-loop!
    pub fn poll<'py>(&self, py: Python<'py>, timeout_ms: Option<u64>) -> PyResult<Bound<'py, PyAny>> {
        // ... fetches data using future_into_py for yielding control back to python asyncio scheduler
        future_into_py(py, async move {
            let records = scanner.poll(timeout).await;
            // ... convert to Python ScanRecords
            Ok(records)
        })
    }
}

## 3. The Goal: Issue #424
The issue asks to support the `async for` loop in Python for `LogScanner`.

```python
async for record in scanner:
    process(record)
```

In Python, this language-level protocol requires the `__aiter__` and `__anext__` magic methods to be implemented.
In PyO3, you implement them directly under the `#[pymethods]` block in Rust.

1. `__aiter__`: Returns the asynchronous iterator (usually `self`).
2. `__anext__`: Asynchronously returns the next item or raises `StopAsyncIteration` when exhausted.

### Step-by-Step Implementation Guide

**Step 1:** Implement `__aiter__` in `bindings/python/src/table.rs` under `#[pymethods] impl LogScanner`.

```rust
fn __aiter__(slf: PyRef<'_, Self>) -> PyRef<'_, Self> {
    slf
}
```

**Step 2:** Implement `__anext__`. This needs to fetch the next record. Because it must return an awaitable just like `poll()` does, use `future_into_py`.

```rust
fn __anext__<'py>(&mut self, py: Python<'py>) -> PyResult<Option<Bound<'py, PyAny>>> {
    // Clone scanner references so the async loop can take ownership of them
    let mut scanner_kind = self.inner.clone();
    
    let fut = future_into_py(py, async move {
        // 1. Await the next record/batch from the engine
        
        // 2. If exhausted, return PyStopAsyncIteration
        // return Err(pyo3::exceptions::PyStopAsyncIteration::new_err("stream exhausted"));
        
        // 3. Otherwise return the converted Python object (e.g., Py<ScanRecord>)
        Ok(py_item)
    })?;
    
    Ok(Some(fut))
}
```

### 🧠 The Core Challenge: State Management
A regular `poll()` call returns a whole **batch** of records (`ScanRecords`). 
An `async for record in scanner:` expects **one record** at a time.

You will need to maintain a state queue or iterator internal to `LogScanner` so that `__anext__` can yield individual `ScanRecord` elements from a cached batch, only traversing network (`await poll()`) when the cache is empty!

## 4. How to Build and Test
You can build the Python wheel via `maturin` and test it directly.

1. Open a terminal and `cd /Users/jaredyu/Desktop/open_source/fluss-rust/bindings/python`
2. Activate your python virtual environment.
3. Run `pip install maturin` (if not already installed).
4. Run `maturin develop` to compile and inject the module into your python environment.
5. Create a `test_scanner.py` and run it!

export PATH="$HOME/.cargo/bin:$PATH" && maturin develop